# Taller 1 Spark — Bloque Yerko

**Curso:** Gestión de datos y tecnologías Big Data
**Integrantes:** Yerko Gallardo, Sebastián, José Ignacio Rodríguez
**Bloque:** entorno con uv y Java, RDDs, unión, transformaciones y conversión a DataFrame

Notebook de desarrollo. Pasos: Y1 configuración · Y2 inspección de CSV · Y3 RDDs y unión · Y4 transformaciones · Y5 mayúsculas y DataFrame.

## 1. Configuración del entorno (Y1)

El proyecto se gestiona con **uv**: Python y los paquetes quedan fijados en `pyproject.toml`, `uv.lock` y `.python-version`.
**Java** es una dependencia de ejecución de Spark (el motor corre en la JVM) y se instala fuera de uv.

| Componente | Versión |
|---|---|
| Python | 3.14 |
| PySpark | 4.3.0 |
| Java (JDK) | 21 (Eclipse Temurin) |

**Preparación (macOS):**

```bash
# Java
brew install --cask temurin@21
echo 'export JAVA_HOME=$(/usr/libexec/java_home -v 21)' >> ~/.zshrc
source ~/.zshrc
java -version

# Proyecto
git clone https://github.com/joserodriguezc/gestiondatosytecnologiasbigdata.git
cd gestiondatosytecnologiasbigdata
uv sync --locked

# Editor
code .        # VS Code: seleccionar el kernel .venv (Python 3.14)
# alternativa: uv run jupyter lab
```

No se usa Spark Connect: el taller requiere RDDs y `SparkContext`, por lo que la sesión se crea en modo clásico.

### 1.1 Verificación del intérprete, versiones y Java

In [ ]:
import os, sys, platform, subprocess
import pyspark

# Si el editor no heredó JAVA_HOME, se obtiene del sistema (macOS) sin fijar rutas personales
if not os.environ.get("JAVA_HOME") and platform.system() == "Darwin":
    os.environ["JAVA_HOME"] = subprocess.run(
        ["/usr/libexec/java_home", "-v", "21"], capture_output=True, text=True
    ).stdout.strip()

print("Sistema    :", platform.system(), platform.machine())
print("Intérprete :", sys.executable)
print("Python     :", platform.python_version())
print("PySpark    :", pyspark.__version__)
print("JAVA_HOME  :", os.environ.get("JAVA_HOME", "NO DEFINIDO"))

java = subprocess.run(["java", "-version"], capture_output=True, text=True)
print("Java       :")
print(java.stderr.strip())   # java -version escribe en stderr

### 1.2 SparkSession y SparkContext

`PYSPARK_PYTHON` y `PYSPARK_DRIVER_PYTHON` apuntan al intérprete del proyecto para que los procesos Python de Spark usen el mismo entorno.

In [ ]:
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Taller1_Spark")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
sc = spark.sparkContext

print("Tipo de sesión     :", type(spark).__module__)   # pyspark.sql.session = modo clásico
print("Versión Spark      :", spark.version)
print("Master             :", sc.master)
print("Nombre aplicación  :", sc.appName)
print("Paralelismo default:", sc.defaultParallelism)
print("Spark UI           :", sc.uiWebUrl)

### 1.3 Prueba mínima de RDD y DataFrame

In [ ]:
rdd_prueba = sc.parallelize(range(1, 11), 2)
print("RDD -> suma:", rdd_prueba.sum(), "| particiones:", rdd_prueba.getNumPartitions())

df_prueba = spark.createDataFrame([(1, "a"), (2, "b")], ["id", "letra"])
df_prueba.show()
df_prueba.printSchema()

## 2. Inspección de archivos de deportistas (Y2)

_Pendiente: requiere `deportista.csv` y `deportista2.csv`._